In [ ]:
# pip install pg8000 urllib3 python-dotenv
# pip install -U certifi urllib3

In [ ]:
import os
import json
import time
import random
from datetime import datetime, timedelta, date
from urllib.parse import quote

import pg8000.native
import urllib3

try:
    # optional
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass


# ============================================================
# CONFIG
# ============================================================
FB_API_VERSION = os.getenv("FB_API_VERSION", "v18.0")
INSIGHTS_BATCH_SIZE = int(os.getenv("INSIGHTS_BATCH_SIZE", "50"))

MAX_RETRIES = int(os.getenv("MAX_RETRIES", "5"))
BASE_BACKOFF = float(os.getenv("BASE_BACKOFF", "1.0"))

HTTP_TIMEOUT_CONNECT = float(os.getenv("HTTP_TIMEOUT_CONNECT", "30"))
HTTP_TIMEOUT_READ = float(os.getenv("HTTP_TIMEOUT_READ", "120"))

SLEEP_BETWEEN_CALLS_MIN = float(os.getenv("SLEEP_BETWEEN_CALLS_MIN", "0.05"))
SLEEP_BETWEEN_CALLS_MAX = float(os.getenv("SLEEP_BETWEEN_CALLS_MAX", "0.25"))


# ============================================================
# UTIL
# ============================================================
def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


def daterange(start: date, end_inclusive: date):
    d = start
    while d <= end_inclusive:
        yield d
        d += timedelta(days=1)


def normalize_act_id(fb_ad_account_id: str) -> str:
    s = (fb_ad_account_id or "").strip()
    return s if s.startswith("act_") else f"act_{s}"


def is_permission_error(err: dict) -> bool:
    # Meta OAuthException permissions error는 보통 code=200
    return bool(err) and err.get("code") == 200


def is_rate_limit_error(err: dict) -> bool:
    if not err:
        return False
    code = err.get("code")
    sub = err.get("error_subcode")
    msg = (err.get("message") or "").lower()
    if code in (4, 17, 32, 613):
        return True
    if sub in (1504022,):
        return True
    if "rate limit" in msg or "too many calls" in msg or "temporarily unavailable" in msg:
        return True
    return False


def is_transient_http_status(status: int) -> bool:
    return status in (408, 429, 500, 502, 503, 504)


def safe_int(v):
    try:
        return int(float(v))
    except Exception:
        return 0


def safe_float(v):
    try:
        return float(v)
    except Exception:
        return 0.0


# ============================================================
# DB
# ============================================================
def get_db_conn():
    return pg8000.native.Connection(
        user=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        host=os.environ["DB_HOST"],
        database=os.environ["DB_NAME"],
        port=5432
    )


def get_ad_accounts(conn):
    rows = conn.run("SELECT fb_ad_account_id FROM ad_account WHERE fb_ad_account_id IS NOT NULL")
    return [str(r[0]).strip() for r in rows if r and r[0] is not None]


def get_ads_by_account(conn, fb_ad_account_id):
    rows = conn.run("""
        SELECT ad_id, fb_ad_id
        FROM ad
        WHERE account_id = (
            SELECT account_id FROM ad_account WHERE fb_ad_account_id = :fb_ad_account_id
        )
        AND fb_ad_id IS NOT NULL
    """, fb_ad_account_id=fb_ad_account_id)
    return [{"ad_id": r[0], "fb_ad_id": str(r[1]).strip()} for r in rows]


def upsert_rows(conn, rows, ad_id_map, created_at_utc):
    """
    rows: list of dict with keys:
      fb_ad_id, date_start, age, gender, reach, impressions, clicks, ctr, inline_post_engagement
    """
    saved = 0
    skipped = 0

    for rec in rows:
        fb_ad_id = rec.get("fb_ad_id")
        if not fb_ad_id or fb_ad_id not in ad_id_map:
            continue

        ad_id = ad_id_map[fb_ad_id]

        try:
            conn.run("""
                INSERT INTO ad_performance_daily
                    (ad_id, date, age, gender, reach, impressions, clicks, ctr,
                     inline_post_engagement, created_at, updated_at)
                VALUES
                    (:ad_id, :date, :age, :gender, :reach, :impressions, :clicks, :ctr,
                     :inline_post_engagement, :created_at, :updated_at)
                ON CONFLICT (ad_id, date, age, gender)
                DO UPDATE SET
                    reach = EXCLUDED.reach,
                    impressions = EXCLUDED.impressions,
                    clicks = EXCLUDED.clicks,
                    ctr = EXCLUDED.ctr,
                    inline_post_engagement = EXCLUDED.inline_post_engagement,
                    updated_at = :updated_at
            """,
            ad_id=ad_id,
            date=rec["date_start"],
            age=rec["age"],
            gender=rec["gender"],
            reach=rec["reach"],
            impressions=rec["impressions"],
            clicks=rec["clicks"],
            ctr=rec["ctr"],
            inline_post_engagement=rec["inline_post_engagement"],
            created_at=created_at_utc,
            updated_at=created_at_utc
            )
            saved += 1
        except Exception as e:
            skipped += 1
            print(f"⚠️ DB upsert failed ad_id={ad_id} fb_ad_id={fb_ad_id} date={rec.get('date_start')}: {e}")

    return saved, skipped


# ============================================================
# META API
# ============================================================
def fetch_insights_batch(http, access_token, act_id, fb_ad_ids, since, until):
    """
    /{act_id}/insights + filtering(IN) 배치 요청 (paging + retry)
    반환: (rows, reason)
      reason: "OK", "PERMISSION", "FAILED"
    """
    base_url = f"https://graph.facebook.com/{FB_API_VERSION}/{act_id}/insights"

    filtering = [{
        "field": "ad.id",
        "operator": "IN",
        "value": fb_ad_ids
    }]

    params = {
        "access_token": access_token,
        "level": "ad",
        "breakdowns": "age,gender",
        "time_range": json.dumps({"since": since, "until": until}),
        "time_increment": "1",
        "fields": "ad_id,ad_name,date_start,reach,impressions,clicks,ctr,inline_post_engagement",
        "filtering": json.dumps(filtering),
        "limit": "5000",
    }

    def build_url(extra=None):
        p = dict(params)
        if extra:
            p.update(extra)
        return base_url + "?" + "&".join([f"{k}={quote(str(v))}" for k, v in p.items()])

    backoff = BASE_BACKOFF

    for attempt in range(MAX_RETRIES):
        try:
            resp = http.request("GET", build_url())
            raw = resp.data.decode("utf-8", errors="replace")
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                sleep_s = backoff + random.uniform(0.0, 0.5)
                print(f"⚠️ Network error {act_id} (attempt {attempt+1}/{MAX_RETRIES}): {e} → sleep {sleep_s:.2f}s")
                time.sleep(sleep_s)
                backoff *= 2
                continue
            return [], "FAILED"

        try:
            data = json.loads(raw)
        except Exception:
            data = {"error": {"message": "json_decode_failed", "raw": raw[:300]}}

        if resp.status == 200 and "data" in data:
            rows = normalize_rows(data["data"], fallback_date=since)

            # paging
            while True:
                paging = data.get("paging") or {}
                cursors = paging.get("cursors") or {}
                after = cursors.get("after")
                if not after:
                    break

                resp2 = http.request("GET", build_url({"after": after}))
                raw2 = resp2.data.decode("utf-8", errors="replace")
                try:
                    data2 = json.loads(raw2)
                except Exception:
                    print("⚠️ paging json decode failed")
                    break

                if resp2.status != 200 or "data" not in data2:
                    print(f"⚠️ paging failed status={resp2.status} body={raw2[:250]}")
                    break

                rows.extend(normalize_rows(data2["data"], fallback_date=since))
                data = data2

            return rows, "OK"

        err = (data.get("error") or {})

        if resp.status == 403 and is_permission_error(err):
            print(f"🚫 Permission denied for {act_id}: {err.get('message')}")
            return [], "PERMISSION"

        if is_rate_limit_error(err) or is_transient_http_status(resp.status):
            if attempt < MAX_RETRIES - 1:
                sleep_s = backoff + random.uniform(0.0, 0.5)
                print(f"⚠️ Transient error {act_id} status={resp.status} (attempt {attempt+1}/{MAX_RETRIES}) "
                      f"→ sleep {sleep_s:.2f}s | {err.get('message')}")
                time.sleep(sleep_s)
                backoff *= 2
                continue

        print(f"⚠️ Non-retryable error {act_id} status={resp.status}: {raw[:350]}")
        return [], "FAILED"

    print(f"⚠️ Retries exhausted for {act_id}.")
    return [], "FAILED"


def normalize_rows(data_rows, fallback_date):
    out = []
    for ins in data_rows or []:
        out.append({
            "fb_ad_id": str(ins.get("ad_id")).strip() if ins.get("ad_id") is not None else None,
            "date_start": ins.get("date_start") or fallback_date,
            "age": ins.get("age") or "unknown",
            "gender": ins.get("gender") or "unknown",
            "reach": safe_int(ins.get("reach", 0)),
            "impressions": safe_int(ins.get("impressions", 0)),
            "clicks": safe_int(ins.get("clicks", 0)),
            "ctr": safe_float(ins.get("ctr", 0)),
            "inline_post_engagement": safe_int(ins.get("inline_post_engagement", 0)),
        })
    return out


# ============================================================
# MAIN BACKFILL
# ============================================================
def backfill(start_date: str, end_date: str):
    """
    start_date/end_date: "YYYY-MM-DD" (inclusive)
    """
    access_token = os.environ["META_ACCESS_TOKEN"]

    start = datetime.strptime(start_date, "%Y-%m-%d").date()
    end = datetime.strptime(end_date, "%Y-%m-%d").date()

    db = get_db_conn()
    http = urllib3.PoolManager(
        timeout=urllib3.Timeout(connect=HTTP_TIMEOUT_CONNECT, read=HTTP_TIMEOUT_READ),
        retries=False
    )

    try:
        accounts = get_ad_accounts(db)
        print(f"✅ DB ad_accounts: {len(accounts)}")

        total_saved = 0
        total_rows = 0
        permission_skipped_accounts = 0

        # 계정별로 광고 목록/맵을 먼저 확보(기간 루프에서 재사용)
        account_ads = []
        for fb_ad_account_id in accounts:
            ads = get_ads_by_account(db, fb_ad_account_id)
            ad_id_map = {a["fb_ad_id"]: a["ad_id"] for a in ads}
            fb_ad_ids = list(ad_id_map.keys())
            account_ads.append((fb_ad_account_id, normalize_act_id(fb_ad_account_id), fb_ad_ids, ad_id_map))
        print("✅ Loaded ads for accounts (from DB).")

        for d in daterange(start, end):
            since = d.strftime("%Y-%m-%d")
            until = (d + timedelta(days=1)).strftime("%Y-%m-%d")
            print(f"\n==============================")
            print(f"📅 Backfill day: {since} (since={since}, until={until})")

            day_rows = 0
            day_saved = 0

            created_at_utc = datetime.utcnow()

            for fb_ad_account_id, act_id, fb_ad_ids, ad_id_map in account_ads:
                if not fb_ad_ids:
                    continue

                # 계정 단위 권한 문제면 이 날짜에서만 스킵이 아니라 "아예 계속 스킵"하고 싶으면,
                # 아래에서 PERMISSION이면 다음 날짜도 계속 스킵하도록 캐시해도 됨.
                # (단순화를 위해: PERMISSION이면 해당 날짜에서 계정 스킵)
                account_permission_denied = False

                for batch in chunk_list(fb_ad_ids, INSIGHTS_BATCH_SIZE):
                    rows, reason = fetch_insights_batch(
                        http=http,
                        access_token=access_token,
                        act_id=act_id,
                        fb_ad_ids=batch,
                        since=since,
                        until=until
                    )

                    if reason == "PERMISSION":
                        account_permission_denied = True
                        break  # 이 계정 배치 더 안 돌림

                    if rows:
                        saved, skipped = upsert_rows(db, rows, ad_id_map, created_at_utc)
                        day_saved += saved
                        total_saved += saved
                        day_rows += len(rows)
                        total_rows += len(rows)

                    time.sleep(random.uniform(SLEEP_BETWEEN_CALLS_MIN, SLEEP_BETWEEN_CALLS_MAX))

                if account_permission_denied:
                    permission_skipped_accounts += 1
                    print(f"🚫 Skip account (permission): {act_id} for day {since}")

            print(f"✅ Day summary {since}: fetched_rows={day_rows}, saved_rows={day_saved}")
            print(f"📌 Running total: fetched_rows={total_rows}, saved_rows={total_saved}")

        print("\n==============================")
        print("✅ Backfill finished.")
        print(f"Total fetched rows: {total_rows}")
        print(f"Total saved rows:   {total_saved}")
        print(f"Permission-skipped account-days: {permission_skipped_accounts}")

    finally:
        try:
            http.clear()
        except Exception:
            pass
        try:
            db.close()
        except Exception:
            pass


if __name__ == "__main__":
    # 예시: 2025-01-01 ~ 2026-01-27 전체 백필
    # 필요에 맞게 바꿔 실행하세요.
    backfill("2026-03-01", "2026-03-29")